In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import datetime as dt
from matplotlib import pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets
from ipywidgets import interact
import matplotlib.ticker as mticker
import seaborn as sns

load_dotenv()

headers = {"Authorization": os.getenv("token_pw")}

def datos_empleados(employee_id, headers, endpoint="planning_units_by_employee"):
    if endpoint == "planning_units_by_employee":
        url = f"https://legacy-api.peopleware.com/v1/employees/{employee_id}/planning_units"
    elif endpoint == "contracts_by_employee":
        url = f"https://legacy-api.peopleware.com/v1/employees/{employee_id}/contracts"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            data_json = response.json()
            if "data" in data_json and isinstance(data_json["data"], list):
                for item in data_json["data"]:
                    item["employee_id"] = employee_id
                return data_json["data"]
            else:
                print(f"Estructura inesperada para {employee_id}")
                return []
        else:
            print(f"Error {response.status_code} en {employee_id}")
            return []
    except Exception as e:
        print(f"Excepción con {employee_id}: {e}")
        return []


endpoints = [
    # "employees",
    "planning_units",
    # "contracts",
    "activities",
    "curves",
    "curve_value_types",
    "curve_versions",
    "curve_event_types",
]

# lista_endpoints_relaciones = ["planning_units_by_employee", "contracts_by_employee"]
lista_endpoints_relaciones = []
empleados_id = []
planning_unit_id = []

def extraer_datos(endpoint):

    base_url = "https://legacy-api.peopleware.com/v1/"
    endpoint = endpoint
    url = f"{base_url}{endpoint}"
    headers = {"Authorization": os.getenv("token_pw"),
            "Content-Type": "application/json"}
    x = requests.get(url, headers=headers)

    if x.status_code == 200:
        data_json = x.json()

        if endpoint == 'employee_current_addresses':
            first_key = list(data_json.keys())[0]
            if isinstance(data_json[first_key], list):
                df = pd.DataFrame(data_json[first_key])
            elif isinstance(data_json[first_key], dict):
                df = pd.DataFrame([data_json[first_key]])
            else:
                df = pd.DataFrame()
        if f"{endpoint}" in data_json and isinstance(data_json[f"{endpoint}"], list):
            df = pd.DataFrame(data_json[f"{endpoint}"])
        elif "data" in data_json and isinstance(data_json["data"], list):
            df = pd.DataFrame(data_json["data"])
        elif isinstance(data_json, dict):
            df = pd.DataFrame([data_json])
        else:
            print("La estructura JSON no se pudo procesar automáticamente para crear el DataFrame.")
            df = pd.DataFrame()

        if not df.empty:
            if endpoint == "employees":
                empleados_id.extend(df['employee_id'].tolist())
            elif endpoint == "planning_units":
                planning_unit_id.extend(df['planning_unit_id'].tolist())

            return df


        else:
            print("No se generó ningún DataFrame para guardar.")

    else:
        print(f"Error al obtener datos: Código de estado {x.status_code}")
        print(x.text)


def datos_peopleware():
    dfs = {}
    for endpoint in endpoints:
        dfs[endpoint] = extraer_datos(endpoint)
        time.sleep(2)


    for endpoint in lista_endpoints_relaciones:
        data_list = []

        with ThreadPoolExecutor(max_workers=30) as executor:
            futures = [executor.submit(datos_empleados, emp_id, headers, endpoint) for emp_id in empleados_id]
            for future in as_completed(futures):
                result = future.result()
                if result:
                    data_list.extend(result)

        df_epr = pd.DataFrame(data_list)

        if not df_epr.empty:
            dfs[endpoint] = df_epr
        else:
            print("No se generó ningún DataFrame para guardar.")
        data_list = []
    return dfs



def pw_requerido(planning_units, dates):

    time_index = pd.date_range(start='00:00', end='23:45', freq='15min').time
    all_rows = []
    for planning_unit_id in planning_units:
        for date in dates:
            start_date = date.strftime('%Y-%m-%d')
            base_url = "https://legacy-api.peopleware.com/v1/"
            endpoint = f"planning_units/{planning_unit_id}/requirements/{start_date}"
            url = f"{base_url}{endpoint}"
            headers = {"Authorization": os.getenv("token_pw"), "Content-Type": "application/json"}
            x = requests.get(url, headers=headers)

            if x.status_code == 200:
                data_json = x.json()
                requirements = data_json.get('requirements', [])
                # print("requirements:", requirements)
                for req in requirements:
                    activity_id = req.get('activity_id')
                    date_val = req.get('date', start_date)
                    planning_unit_id_val = req.get('planning_unit_id', planning_unit_id)
                    raster = req.get('raster')
                    values = req.get('values', [])
                    if raster == 900 and len(values) == len(time_index):
                        for t, v in zip(time_index, values):
                            all_rows.append({
                                'time': t,
                                'activity_id': activity_id,
                                'date': date_val,
                                'planning_unit_id': planning_unit_id_val,
                                'value': v
                            })
                            # time.sleep(0.5)
                    elif raster == 1800 and len(values) * 2 == len(time_index):
                        expanded_values = [v for v in values for _ in range(2)]
                        for t, v in zip(time_index, expanded_values):
                            all_rows.append({
                                'time': t,
                                'activity_id': activity_id,
                                'date': date_val,
                                'planning_unit_id': planning_unit_id_val,
                                'value': v
                            })
                            # time.sleep(0.5)
                    else:
                        print(f"planning_unit_id {planning_unit_id_val} tiene {len(values)} valores y raster {raster}, pero se esperaban {len(time_index)}. Ignorado.")
            else:
                print(f"Error al obtener datos: Código de estado {x.status_code}")
                print(x.text)
    if all_rows:
        df_result = pd.DataFrame(all_rows)
        return df_result
    else:
        print("No se generó ningún DataFrame para guardar o el resultado estaba vacío.")
        return None


def pw_forecast(queues, dates):

    time_index = pd.date_range(start='00:00', end='23:45', freq='15min').time
    all_rows = []
    for curve_id in queues:
        for date in dates:
            start_date = date.strftime('%Y-%m-%d')
            base_url = "https://legacy-api.peopleware.com/v1/"
            endpoint = f"curves/{curve_id}/values/{start_date}"
            url = f"{base_url}{endpoint}"
            headers = {"Authorization": os.getenv("token_pw"), "Content-Type": "application/json"}
            x = requests.get(url, headers=headers)

            if x.status_code == 200:
                data_json = x.json()
                # print(data_json)
                requirements = data_json.get('curve_values', [])
                # print("requirements:", requirements)
                for req in requirements:
                    curve_event_type_id = req.get('curve_value_type_id')
                    curve_value_type_id = req.get('curve_value_type_id')
                    curve_version_id = req.get('curve_version_id')
                    date_val = req.get('date', date)
                    curve_id = req.get('curve_id', curve_id)
                    raster = req.get('raster')
                    values = req.get('values', [])
                    if raster == 900 and len(values) == len(time_index):
                        
                        for t, v in zip(time_index, values):
                            all_rows.append({
                                'time': t,
                                'curve_event_type_id': curve_event_type_id,
                                'curve_value_type_id': curve_value_type_id,
                                'curve_version_id': curve_version_id,
                                'date': date,
                                'curve_id': curve_id,
                                'value': v
                            })
                            # time.sleep(0.5)
                    elif raster == 1800 and len(values) * 2 == len(time_index):
                        expanded_values = [v for v in values for _ in range(2)]
                        for t, v in zip(time_index, expanded_values):
                            all_rows.append({
                                'time': t,
                                'curve_event_type_id': curve_event_type_id,
                                'curve_value_type_id': curve_value_type_id,
                                'curve_version_id': curve_version_id,
                                'date': date_val,
                                'curve_id': curve_id,
                                'value': v
                            })
                            # time.sleep(0.5)
                    else:
                        print(f"curve_id {curve_id} tiene {len(values)} valores y raster {raster}, pero se esperaban {len(time_index)}. Ignorado.")
            else:
                print(f"Error al obtener datos: Código de estado {x.status_code}")
                print(x.text)
    if all_rows:
        df_result = pd.DataFrame(all_rows)
        return df_result
    else:
        print("No se generó ningún DataFrame para guardar o el resultado estaba vacío.")
        return None




In [ ]:
dfs_peopleware = datos_peopleware()
dfs_peopleware.keys()

In [ ]:
for key, df in dfs_peopleware.items():
    if df is not None and not df.empty:
        df.to_csv(f"./csvs_peopleware/{key}.csv", index=False)

In [ ]:
hoy = dt.date.today()
inicio = hoy - dt.timedelta(days=7)
fin = hoy + dt.timedelta(days=7)
rango_fechas = pd.date_range(start=inicio, end=fin)

In [ ]:
df_planning_units = dfs_peopleware['planning_units']
planning_units_ids = df_planning_units['planning_unit_id'].unique()
df_requerido = pd.DataFrame(pw_requerido(planning_units_ids, rango_fechas))

df_base_req = df_requerido.merge(df_planning_units[['planning_unit_id', 'name']], on='planning_unit_id', how='left').drop(columns=['planning_unit_id', 'activity_id'])
df_base_req['date'] = pd.to_datetime(df_base_req['date'])
df_base_req['time'] = pd.to_timedelta(df_base_req['time'].apply(lambda x: f"{x:02d}:00:00"))
df_base_req['fecha_hora'] = df_base_req['date'] + df_base_req['time']
df_base_req.rename(columns={'name': 'unidad',
                            'time' : 'hora',
                            'date' : 'fecha',
                        'value': 'requerido'}, inplace=True)
# df_base_req.head()
df_base_req.to_csv('requerido.csv', index=False)

In [ ]:

df_base = pd.read_csv('requerido.csv')
d8 = df_base.copy()

d8['fecha'] = pd.to_datetime(d8['fecha'], errors='coerce').dt.floor('D')

df_agrupado = (
    d8.groupby(['unidad', 'fecha'], as_index=False)['requerido']
    .sum()
    .sort_values(['fecha', 'unidad'])
)

df_agrupado['requerido'] = df_agrupado['requerido'] / 4


df_agrupado['fecha'] = pd.to_datetime(df_agrupado['fecha']).dt.normalize()

tabla_salida = df_agrupado.copy()
tabla_salida.to_csv('requerido_peopleware.csv', index=False)

unidades = sorted(df_agrupado['unidad'].dropna().unique())

@interact(unidad=widgets.Dropdown(
    options=['Todas'] + unidades,
    value='Todas',
    description='Unidad:'
))
def plot_unidad(unidad):
    if unidad == 'Todas':
        plot_df = (
            df_agrupado.groupby('fecha', as_index=False)['requerido']
            .sum()
            .sort_values('fecha')
        )
        hue = None
        titulo = 'Requerido total (todas las unidades)'
    else:
        plot_df = df_agrupado[df_agrupado['unidad'] == unidad].copy()
        hue = None
        titulo = f'Requerido para {unidad}'


    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=plot_df,
        x='fecha',
        y='requerido',
        hue=hue,
        marker='o',
        ax=ax
    )

    # Formato eje X: sólo fecha
    # Opción recomendada: ticks exactamente en tus fechas
    fechas_ticks = sorted(plot_df['fecha'].drop_duplicates())
    ax.xaxis.set_major_locator(mticker.FixedLocator(mdates.date2num(fechas_ticks)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))

    # opcional para que no agregue margen raro
    xmin = min(fechas_ticks) - pd.Timedelta(days=0.35)
    xmax = max(fechas_ticks) + pd.Timedelta(days=0.35)
    ax.set_xlim(xmin, xmax)

    ymin = plot_df['requerido'].min()
    ymax = plot_df['requerido'].max()
    pad = (ymax - ymin) * 0.12 if ymax > ymin else max(1, ymax * 0.12)

    ax.set_ylim(ymin - pad, ymax + pad)

    for _, row in plot_df.iterrows():
        ax.annotate(
            f"{row['requerido']:.0f}",
            xy=(row['fecha'], row['requerido']),
            xytext=(0, 6),
            textcoords='offset points',
            ha='center',
            fontsize=8
        )

    ax.set_title(f'{titulo} dos semanas antes y después de hoy')
    ax.set_xlabel('Fecha')
    ax.set_ylabel('Requerido')


    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.grid()
    plt.show()


In [ ]:
df_forecast = pd.DataFrame(pw_forecast(planning_units_ids, rango_fechas))
df_forecast.head().to_string()